# Foldy–Wouthuysen Derivation: The $g_{\lambda\mu\nu}$ SME Coefficient

**MSc Research — Oyewo Temidayo Solomon, University of Ibadan**

Completes the SME$\to$DM mapping. $g_{\lambda\mu\nu}$ is the fourth and last
coefficient that generates spin-dependent non-relativistic potentials; $b_\mu$,
$H_{\mu\nu}$ and $d_{\mu\nu}$ are treated in the companion notebooks.

Conventions follow `dirac_algebra.py`: Dirac/Pauli representation, metric
$(+,-,-,-)$, $\hbar=c=1$, $\gamma_5=i\gamma^0\gamma^1\gamma^2\gamma^3$.

Reference result: Kostelecký & Lane, *Nonrelativistic Quantum Hamiltonian for
Lorentz Violation*, J. Math. Phys. **40**, 6245 (1999).

## 1. Setup — Import Dirac Algebra

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import sympy as sp
from sympy import I, Matrix, symbols, simplify, Rational, eye, zeros

from dirac_algebra import (beta, gamma, gamma5, alpha, Sigma, sigma_munu,
                           even_part, odd_part, upper, lower,
                           is_even, is_odd, bcf_expansion)

I4, Z4 = eye(4), zeros(4, 4)
p1, p2, p3, m = symbols('p_1 p_2 p_3 m', real=True)
p_vec = [p1, p2, p3]

# bookkeeping: eps tracks order in g, lam tracks order in momentum
eps = symbols('varepsilon', positive=True)
lam = symbols('lambda_', positive=True)

def trunc_g(M):
    return sp.Matrix(sp.expand(M)).applyfunc(
        lambda x: sp.expand(sp.series(sp.expand(x), eps, 0, 2).removeO()))

def keep_leading_p(M):
    M = sp.Matrix(M).subs({p1: lam*p1, p2: lam*p2, p3: lam*p3})
    return sp.simplify(M.applyfunc(
        lambda x: sp.expand(sp.series(sp.expand(x), lam, 0, 2).removeO())).subs(lam, 1))

gamma_lower = [gamma[0], -gamma[1], -gamma[2], -gamma[3]]
print("Dirac algebra loaded; gamma^0 == beta:", simplify(gamma[0]-beta) == Z4)

Dirac algebra loaded; gamma^0 == beta: True


## 2. The $g_{\lambda\mu\nu}$ Lagrangian and Modified Dirac Equation

From Kostelecký & Lane Eq. (1), the kinetic operator is

$$\Gamma_\nu=\gamma_\nu+c_{\mu\nu}\gamma^\mu+d_{\mu\nu}\gamma_5\gamma^\mu
+e_\nu+if_\nu\gamma_5+\tfrac12 g_{\lambda\mu\nu}\sigma^{\lambda\mu}.$$

$g_{\lambda\mu\nu}$ is therefore a **kinetic-sector** coefficient, like
$c_{\mu\nu}$ and $d_{\mu\nu}$ and unlike the mass-sector $b_\mu$ and
$H_{\mu\nu}$. It is antisymmetric in its first two indices, since
$\sigma^{\lambda\mu}$ is; the third index is the derivative index. It is
CPT-odd and dimensionless.

In [2]:
# independent components g_{lambda mu nu} with lambda < mu
gsym = {}
for L_ in range(4):
    for M_ in range(L_+1, 4):
        for N_ in range(4):
            gsym[(L_, M_, N_)] = symbols(f'g_{L_}{M_}{N_}', real=True)

def g(L_, M_, N_):
    if L_ == M_:
        return sp.Integer(0)
    return gsym[(L_, M_, N_)] if L_ < M_ else -gsym[(M_, L_, N_)]

def Gamma_lower(nu, families=None):
    # Gamma_nu = gamma_nu + (1/2) eps g_{lambda mu nu} sigma^{lambda mu}
    M_ = gamma_lower[nu]
    for L_ in range(4):
        for MM in range(4):
            if L_ == MM:
                continue
            key = (min(L_, MM), max(L_, MM), nu)
            if families is not None and key not in families:
                continue
            M_ = M_ + Rational(1, 2)*eps*g(L_, MM, nu)*sigma_munu(L_, MM)
    return M_

dGamma0 = simplify(Gamma_lower(0) - gamma_lower[0])
print("Gamma_0 - gamma_0 involves:",
      sorted({str(x) for x in dGamma0.free_symbols if str(x).startswith('g_')}))

Gamma_0 - gamma_0 involves: ['g_010', 'g_020', 'g_030', 'g_120', 'g_130', 'g_230']


**Result:** only the $\nu=0$ slice of $g$ enters $\Gamma_0$, as expected for a
kinetic-sector coefficient. Because $\Gamma_0\neq\gamma_0$, the naive step of
multiplying by $\gamma^0$ and reading off $E\psi=H\psi$ is not available; the
field redefinition of Sec. 3 is required.

## 3. Field Redefinition and the Relativistic Hamiltonian

Kostelecký & Lane remove the time-derivative coupling with $\psi=A\chi$,

$$A=1-\tfrac12\gamma^0(\Gamma_0-\gamma_0),\qquad
\bar A=1-\tfrac12(\Gamma_0-\gamma_0)\gamma^0,$$

after which

$$H=-\gamma^0\bar A\,\Gamma_j A\,p^j+\gamma^0\bar A M A .$$

In [3]:
A    = trunc_g(I4 - Rational(1,2)*gamma[0]*dGamma0)
Abar = trunc_g(I4 - Rational(1,2)*dGamma0*gamma[0])

def relativistic_H(families=None):
    dG = simplify(Gamma_lower(0, families) - gamma_lower[0])
    A_    = trunc_g(I4 - Rational(1,2)*gamma[0]*dG)
    Abar_ = trunc_g(I4 - Rational(1,2)*dG*gamma[0])
    H = gamma[0]*trunc_g(Abar_*(m*I4)*A_)
    for j in (1, 2, 3):
        H = H - gamma[0]*trunc_g(Abar_*Gamma_lower(j, families)*A_)*p_vec[j-1]
    return sp.Matrix(trunc_g(H))

H_rel = relativistic_H()
# K&L choose A so the *Lagrangian's* time-derivative coupling becomes standard,
# i.e.  Abar Gamma_0 A = gamma_0  -- not  A^dag A = gamma^0 Gamma_0.
print("Abar Gamma_0 A == gamma_0  (K&L criterion) :",
      simplify(trunc_g(Abar*Gamma_lower(0)*A - gamma_lower[0])) == Z4)
print("H Hermitian                                :",
      simplify(trunc_g(H_rel.conjugate().T - H_rel)) == Z4)

Abar Gamma_0 A == gamma_0  (K&L criterion) : True
H Hermitian                                : True


## 4. FW Transformation

Split $H=\beta m+\mathcal E+\mathcal O$ and remove the odd part with
$S=-\tfrac{i}{2m}\beta\mathcal O$, using the BCH expansion in
`dirac_algebra.bcf_expansion`.

In [4]:
def fw_blocks(families=None):
    H = relativistic_H(families)
    O = sp.Matrix(odd_part(H))
    S = sp.Matrix(simplify(-I*beta*O/(2*m)))
    _, total = bcf_expansion(S, H, order=2)
    E = even_part(sp.Matrix(trunc_g(sp.Matrix(total))))
    def strip(B):
        B = sp.Matrix(B)
        return keep_leading_p(sp.Matrix(sp.expand(B) - sp.expand(B.subs(eps, 0))).subs(eps, 1))
    return strip(upper(E)), strip(lower(E))

H_particle, H_lower = fw_blocks()
print("Non-relativistic Hamiltonian, particle (upper) block, O(g):")
sp.pprint(simplify(H_particle))

Non-relativistic Hamiltonian, particle (upper) block, O(g):


⎡                                    g₀₁₀⋅p₂ - g₀₂₀⋅p₁ - g₁₂₀⋅m - g₁₂₁⋅p₁ - g₁ ↪
⎢                                                                              ↪
⎣-ⅈ⋅g₀₁₀⋅p₃ + g₀₂₀⋅p₃ + ⅈ⋅g₀₃₀⋅p₁ - g₀₃₀⋅p₂ + ⅈ⋅g₁₃₀⋅m + ⅈ⋅g₁₃₁⋅p₁ + ⅈ⋅g₁₃₂⋅p₂ ↪

↪ ₂₂⋅p₂ - g₁₂₃⋅p₃                                      ⅈ⋅g₀₁₀⋅p₃ + g₀₂₀⋅p₃ - ⅈ ↪
↪                                                                              ↪
↪  + ⅈ⋅g₁₃₃⋅p₃ - g₂₃₀⋅m - g₂₃₁⋅p₁ - g₂₃₂⋅p₂ - g₂₃₃⋅p₃                          ↪

↪ ⋅g₀₃₀⋅p₁ - g₀₃₀⋅p₂ - ⅈ⋅g₁₃₀⋅m - ⅈ⋅g₁₃₁⋅p₁ - ⅈ⋅g₁₃₂⋅p₂ - ⅈ⋅g₁₃₃⋅p₃ - g₂₃₀⋅m - ↪
↪                                                                              ↪
↪             -g₀₁₀⋅p₂ + g₀₂₀⋅p₁ + g₁₂₀⋅m + g₁₂₁⋅p₁ + g₁₂₂⋅p₂ + g₁₂₃⋅p₃        ↪

↪  g₂₃₁⋅p₁ - g₂₃₂⋅p₂ - g₂₃₃⋅p₃⎤
↪                             ⎥
↪                             ⎦


## 5. Decomposition by Index Family

Write $\tilde g_j\equiv\tfrac12\varepsilon_{jkl}g_{kl0}$ for the dual of the
spatial–spatial $\nu=0$ block and $(\mathbf g_{00})_k\equiv g_{k00}$. Then

$$\begin{aligned}
H_{\rm NR}(g_{[kl]0})&=-m\,\tilde{\mathbf g}\cdot\boldsymbol\sigma &&\to V_2,\\
H_{\rm NR}(g_{[0k]0})&=+\boldsymbol\sigma\cdot(\mathbf p\times\mathbf g_{00}) &&\to V_7,\\
H_{\rm NR}(g_{[kl]j})&=+\tfrac12\varepsilon_{klm}g_{mlj}\,p^j\,\sigma^k &&\to V_7,V_8,
\end{aligned}$$

with $g_{[0k]j}$ contributing nothing at this order. Here $\mathbf p$ is the physical momentum, $p^j$; the next cell checks these three expressions against the derivation.

In [5]:
s2 = [sp.Matrix([[0,1],[1,0]]), sp.Matrix([[0,-I],[I,0]]), sp.Matrix([[1,0],[0,-1]])]
def lc(i, j, k):
    return Rational(1,2)*(i-j)*(j-k)*(k-i)

families = {
    "g_[kl]0  (spatial-spatial, nu=0)": {(a,b,0) for a,b in [(1,2),(1,3),(2,3)]},
    "g_[0k]0  (temporal-spatial, nu=0)": {(0,a,0) for a in (1,2,3)},
    "g_[kl]j  (spatial-spatial, nu=j)": {(a,b,j) for a,b in [(1,2),(1,3),(2,3)] for j in (1,2,3)},
    "g_[0k]j  (temporal-spatial, nu=j)": {(0,a,j) for a in (1,2,3) for j in (1,2,3)},
}
for name, fam in families.items():
    up_, _ = fw_blocks(fam)
    print(f"{name}: {'vanishes at this order' if up_ == sp.zeros(2,2) else 'contributes'}")

g_[kl]0  (spatial-spatial, nu=0): contributes


g_[0k]0  (temporal-spatial, nu=0): contributes


g_[kl]j  (spatial-spatial, nu=j): contributes


g_[0k]j  (temporal-spatial, nu=j): vanishes at this order


In [6]:
# Check the family formulas quoted above (p = physical momentum p^j) against the derivation
from sympy import LeviCivita as E
S2 = [sp.Matrix([[0,1],[1,0]]), sp.Matrix([[0,-I],[I,0]]), sp.Matrix([[1,0],[0,-1]])]
dot = lambda v: sum((v[k]*S2[k] for k in range(3)), sp.zeros(2,2))
g00 = [g(k,0,0) for k in (1,2,3)]
gt  = [Rational(1,2)*sum(E(j,k,l)*g(k+1,l+1,0) for k in range(3) for l in range(3)) for j in range(3)]
quoted = {
  "g_[kl]0": -m*dot(gt),
  "g_[0k]0": dot([sum(E(k,a,b)*p_vec[a]*g00[b] for a in range(3) for b in range(3)) for k in range(3)]),
  "g_[kl]j": Rational(1,2)*dot([sum(E(k,l,n)*g(n+1,l+1,j+1)*p_vec[j] for l in range(3) for n in range(3) for j in range(3)) for k in range(3)]),
}
for name, formula in quoted.items():
    key = [k for k in families if k.startswith(name)][0]
    hp, _ = fw_blocks(families[key])
    print(f"{name}: quoted formula equals derivation -> {simplify(sp.expand(hp - formula)) == sp.zeros(2,2)}")

g_[kl]0: quoted formula equals derivation -> True


g_[0k]0: quoted formula equals derivation -> True


g_[kl]j: quoted formula equals derivation -> True


## 6. Comparison with Kostelecký & Lane

Their non-relativistic Hamiltonian contains, at the two leading orders,

$$h\supset-\tfrac12 m\,\varepsilon_{jkl}g_{kl0}\sigma^j
-m\,\varepsilon_{klm}\Big(\tfrac12 g_{mlj}+g_{m00}\delta_{jl}\Big)\frac{p_j}{m}\sigma^k .$$

Their lower-index $p_j$ is the **covariant** component $p_j=-p^j$, fixed by
their own free-particle term $m\mathcal P_0:=-p_j\gamma^0\gamma^j$, which
equals $-\boldsymbol\alpha\cdot\mathbf p$ and so requires $p_j=-p^j$ to
recover $+\boldsymbol\alpha\cdot\mathbf p+\beta m$. The same convention was
established independently for $d_{\mu\nu}$.

In [7]:
def KL_target(p_covariant=True):
    q = [-x for x in p_vec] if p_covariant else p_vec
    T = sp.zeros(2, 2)
    for j in range(3):
        for k in range(3):
            for l in range(3):
                c = lc(j, k, l)
                if c != 0:
                    T += -Rational(1,2)*m*c*g(k+1, l+1, 0)*s2[j]
    for k in range(3):
        for l in range(3):
            for M_ in range(3):
                c = lc(k, l, M_)
                if c == 0:
                    continue
                for j in range(3):
                    coef = Rational(1,2)*g(M_+1, l+1, j+1) + (g(M_+1, 0, 0) if j == l else 0)
                    T += -c*coef*q[j]*s2[k]
    return simplify(T)

for label, cov in [("p_j covariant (-p^j)", True), ("p_j physical (+p^j)", False)]:
    diff = simplify(sp.expand(H_particle - KL_target(cov)))
    print(f"  {label:24s} matches Kostelecky & Lane ? {diff == sp.zeros(2,2)}")

  p_j covariant (-p^j)     matches Kostelecky & Lane ? True


  p_j physical (+p^j)      matches Kostelecky & Lane ? False


## 7. Matter–Antimatter Behaviour

In hole theory, an antiparticle of momentum $\mathbf p$ and spin $\mathbf s$ is the absence of a negative-energy solution with momentum $-\mathbf p$ and spin $-\mathbf s$, and has the opposite energy. A CPT test compares the particle with its CPT-conjugate state: an antiparticle with the same momentum and reversed spin. Labelling that state by the particle's spin operator gives

$$H^{\bar f}_{NR}(\mathbf p,\boldsymbol\sigma) = -H_{\rm lower}(-\mathbf p,\boldsymbol\sigma).$$

The momentum reversal matters. Two of the three families are linear in $\mathbf p$, and omitting the reversal gives the wrong answer for them. The derivation for every SME coefficient is in `FW_antiparticle_all_coefficients.ipynb`.

In [8]:
flipP = lambda M: M.subs({p1: -p1, p2: -p2, p3: -p3}, simultaneous=True)
print(f"{'family':36s} {'CPT-conjugate antiparticle':28s} relation")
for name, fam in families.items():
    hp, hl = fw_blocks(fam)
    if hp == sp.zeros(2,2):
        print(f"  {name:34s} {'(no contribution)':28s} --")
        continue
    ha = simplify(-flipP(hl))                      # antiparticle, same p, reversed spin
    same = simplify(sp.expand(ha - hp)) == sp.zeros(2,2)
    flip = simplify(sp.expand(ha + hp)) == sp.zeros(2,2)
    verdict = "SAME" if same else ("FLIPPED" if flip else "neither")
    rel = "g_fbar = g_f" if same else ("g_fbar = -g_f" if flip else "?")
    print(f"  {name:34s} {verdict:28s} {rel}")

family                               CPT-conjugate antiparticle   relation


  g_[kl]0  (spatial-spatial, nu=0)   FLIPPED                      g_fbar = -g_f


  g_[0k]0  (temporal-spatial, nu=0)  FLIPPED                      g_fbar = -g_f


  g_[kl]j  (spatial-spatial, nu=j)   FLIPPED                      g_fbar = -g_f


  g_[0k]j  (temporal-spatial, nu=j)  (no contribution)            --


**Result.** All three contributing families flip sign between CPT-conjugate matter and antimatter states, as the CPT-odd character of $g_{\lambda\mu\nu}$ requires. At the same physical spin, all three give identical particle and antiparticle shifts, in line with Kostelecký & Lane's antiparticle rule $g\to+g$. The signature is set by the coefficient's CPT parity, exactly as for $b_\mu$.

## 8. Summary

| family | $H_{\rm NR}$ | DM potential | $1/m$ order | matter–antimatter (CPT-conjugate) |
|---|---|---|---|---|
| $g_{[kl]0}$ | $-m\,\tilde{\mathbf g}\cdot\boldsymbol\sigma$ | $V_2$ | $m^{+1}$ | $g^{\bar f}=-g^f$ |
| $g_{[0k]0}$ | $+\boldsymbol\sigma\cdot(\mathbf p\times\mathbf g_{00})$ | $V_7$ | $m^{0}$ | $g^{\bar f}=-g^f$ |
| $g_{[kl]j}$ | $+\tfrac12\varepsilon_{klm}g_{mlj}p^j\sigma^k$ | $V_7,V_8$ | $m^{0}$ | $g^{\bar f}=-g^f$ |
| $g_{[0k]j}$ | — | — | — | no contribution at this order |

All three contributing families reproduce Kostelecký & Lane term by term.